# Question 4: Final Passage Analysis & Speed Demon Benchmark\n\nThis notebook contains the final Speed Demon benchmarking for the integrated pipeline and the comparative analysis report as required by the assignment.\n

In [ ]:
import sys\nimport os\nimport time\nsys.path.append(os.path.abspath(os.path.join('..')))\n\nfrom src.streamlit_app import load_models\n\nprint('Loading all shared NLP models...')\n(q1_vocab, q1_log_prob, hmm_tagset, emit_lp, trans_lp, \n q3_delete_index, q3_unigram, pcfg_parser, score_bi, score_tri) = load_models()\nprint('Models loaded successfully.')\n

## 1. Speed Demon Benchmark\nWe generate exactly 1,000 simulated words (some merged, some misspelled) and measure latency.\n

In [ ]:
import random\nfrom src.q1_funcs import viterbi_segment\nfrom src.q3_funcs import method_b_candidates, best_unigram_candidate\n\n# Generate 1,000 simulated words from vocab\nrandom.seed(42)\nvocab_list = list(q1_vocab)\nsimulated_stream = []\nfor _ in range(1000):\n    w = random.choice(vocab_list)\n    if random.random() < 0.1: # 10% chance to merge two words\n        w = w + random.choice(vocab_list)\n    elif random.random() < 0.1: # 10% chance to misspell (delete a char)\n        if len(w) > 2:\n            idx = random.randint(0, len(w)-1)\n            w = w[:idx] + w[idx+1:]\n    simulated_stream.append(w)\n\nprint(f'Generated {len(simulated_stream)} words for benchmark.')\n\nstart_time = time.time()\nprocessed_count = 0\nfor token in simulated_stream:\n    if token in q1_vocab:\n        pass\n    else:\n        segments = viterbi_segment(token, q1_vocab, q1_log_prob)\n        if len(segments) > 1:\n            pass # Handled by segmenter\n        else:\n            candidates = method_b_candidates(token, q3_delete_index)\n            best = best_unigram_candidate(candidates, q3_unigram)\n    processed_count += 1\n\ntotal_time_ms = (time.time() - start_time) * 1000\nprint(f'Total Time for 1000 words (Full Check): {total_time_ms:.2f} ms')\nprint(f'Average per-word latency: {total_time_ms/1000:.2f} ms')\n

## 2. Comparative Analysis Report\n\n### System Interactions\nWhen tokens are merged or misspelled, segmentation and spelling alerts correct them before they reach the PCFG parser. Without these upstream corrections, the PCFG would immediately fail parsing the sentence due to out-of-vocabulary tokens or impossible tag sequences. Correcting spelling errors often completely changes the chosen method downstream because a sentence goes from 'Ungrammatical' (unparseable) to 'Grammatical' (parsed).\n\n### Speed Demon Results & Conclusion\nAs seen in the benchmark, the average per-token latency for running both the segmentation check (Viterbi) and the spelling check (Symmetric Delete) is generally on the order of a few milliseconds. Since average human typing speed is ~40-80 words per minute (i.e., one word every ~750ms), the segmentation + spelling layer is **absolutely fast enough to run live per-token**. There is no need to throttle it to the grammar check interval.\n\n### PCFG vs N-gram Agreements\nThe PCFG detects global structural errors (e.g., missing main verbs, unclosed clauses) but has a strict, limited lexicon. The N-gram models detect local sequence anomalies (e.g., 'the an cat') but fail to capture long-distance agreement. They often disagree when a sentence is structurally valid but uses rare word combinations (PCFG accepts, N-gram rejects), or when a sentence is locally smooth but structurally broken (N-gram accepts, PCFG rejects).\n